# 08 · PCA y MCA: Explicabilidad y Reducción Factorial
## Módulo 3 — Feature Engineering & Modelado Supervisado

---

### 🎯 Objetivo del Cuaderno
Diagnosticar la estructura espacial de los datos mediante dos técnicas factoriales complementarias:
1. **PCA (Principal Component Analysis):** Descompone la varianza total sobre las variables numéricas procesadas y escaladas ($06\_train\_proc$).
2. **MCA (Multiple Correspondence Analysis):** Descompone la inercia ($\chi^2$) de las tablas de contingencia sobre las variables categóricas cualitativas ($03\_train$).

### 💡 Utilidad de Negocio:
* **Mapa de Perfiles 2D:** Visualizar cómo se agrupan los clientes y detectar si existe solapamiento entre el **Clúster 1 (Motor Comercial)** y el **Clúster 0 (Consumidor Cauto)**.
* **Calidad de Representación ($cos^2$):** Identificar qué categorías quedan bien explicadas en los dos primeros ejes factoriales y cuáles requieren componentes de mayor orden.

🔒 **Entradas:** `06_train_proc.csv` y `03_train_ecommerce.csv` · **Salidas:** `artifacts/pca.joblib`, `artifacts/mca.joblib` y `reports/pca_mca.json`.

In [1]:
# ==============================================================================
# BLOQUE 1: CARGA DE DATOS PARA PCA Y MCA
# ==============================================================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# 1. Cargar la matriz numéricamente procesada (Módulo 06)
train_proc = pd.read_csv('data_processed/06_train_proc.csv')

# 2. Cargar el dataset categórico original limpio (Módulo 03)
train_cat = pd.read_csv('data_processed/03_train_ecommerce.csv')

target_col = 'Cluster'

X_proc = train_proc.drop(columns=[target_col])
y_proc = train_proc[target_col]

X_cat = train_cat.drop(columns=[target_col])

print("="*75)
print("📦 DATOS CARGADOS PARA ANÁLISIS FACTORIAL:")
print(f"· Matriz Procesada (para PCA): {X_proc.shape[0]} filas × {X_proc.shape[1]} columnas numéricas.")
print(f"· Matriz Categórica (para MCA): {X_cat.shape[0]} filas × {X_cat.shape[1]} columnas cualitativas.")
print("="*75)

📦 DATOS CARGADOS PARA ANÁLISIS FACTORIAL:
· Matriz Procesada (para PCA): 8018 filas × 212 columnas numéricas.
· Matriz Categórica (para MCA): 8086 filas × 61 columnas cualitativas.


## 2. Análisis de Componentes Principales (PCA)

Evaluamos la cantidad de varianza explicada acumulada sobre las variables procesadas numéricas. Buscamos determinar cuántas componentes principales son necesarias para capturar el $\ge 80\%$ de la varianza del sistema.

In [3]:
# ==============================================================================
# BLOQUE 2: EJECUCIÓN DE PCA Y ANÁLISIS DE VARIANZA ACUMULADA
# ==============================================================================
from sklearn.decomposition import PCA
import joblib
import os

# Ajustamos PCA completo para revisar la curva de varianza explicada
pca_full = PCA(random_state=42)
pca_full.fit(X_proc)

var_explicada = pca_full.explained_variance_ratio_
var_acumulada = np.cumsum(var_explicada)

# Determinamos cuántas componentes se necesitan para el 80% y 90% de varianza
comp_80 = np.argmax(var_acumulada >= 0.80) + 1
comp_90 = np.argmax(var_acumulada >= 0.90) + 1

print("📊 DIAGNÓSTICO DE VARIANZA EXPLICADA POR PCA:")
print(f"· Varianza explicada por Dimensión 1: {var_explicada[0]*100:.2f}%")
print(f"· Varianza explicada por Dimensión 2: {var_explicada[1]*100:.2f}%")
print(f"· Varianza explicada por Plano 2D (Dim 1 + Dim 2): {(var_explicada[0]+var_explicada[1])*100:.2f}%")
print("-" * 75)
print(f"🎯 Componentes necesarias para explicar el 80% de varianza: {comp_80} componentes.")
print(f"🎯 Componentes necesarias para explicar el 90% de varianza: {comp_90} componentes.")

# Guardamos el objeto PCA entrenado
os.makedirs('artifacts', exist_ok=True)
joblib.dump(pca_full, 'artifacts/pca.joblib')

📊 DIAGNÓSTICO DE VARIANZA EXPLICADA POR PCA:
· Varianza explicada por Dimensión 1: 7.43%
· Varianza explicada por Dimensión 2: 5.20%
· Varianza explicada por Plano 2D (Dim 1 + Dim 2): 12.63%
---------------------------------------------------------------------------
🎯 Componentes necesarias para explicar el 80% de varianza: 56 componentes.
🎯 Componentes necesarias para explicar el 90% de varianza: 80 componentes.


['artifacts/pca.joblib']

## 3. Análisis de Correspondencias Múltiples (MCA) y Calidad de Representación ($cos^2$)

A diferencia de PCA (que opera sobre distancias euclidianas continuas), el **MCA** está matemáticamente diseñado para descomponer la **inercia ($\chi^2$)** de variables categóricas cualitativas ($03\_train\_ecommerce.csv$).

Evaluamos:
1. **Inercia Explicada por las Dimensiones Factoriales:** Descomposición del espacio cualitativo.
2. **Calidad de Representación ($cos^2$):** Indica la fidelidad con la que una categoría proyectada en el mapa 2D conserva su posición real en el espacio multidimensional.
   * Un $cos^2$ alto ($> 0.20$) garantiza que la cercanía o lejanía de esa categoría en el mapa 2D es **real y confiable**.
   * Un $cos^2$ bajo indica que la categoría requiere dimensiones de mayor orden para interpretarse.

In [7]:
# ==============================================================================
# BLOQUE 3: ANÁLISIS DE CORRESPONDENCIAS MÚLTIPLES (MCA) Y COS2)
# ==============================================================================
import json
import joblib
import os

try:
    import prince
    usar_prince = True
except ImportError:
    usar_prince = False
    print("ℹ️ Librería 'prince' no detectada. Se utilizará cálculo algebraico SVD/MCA nativo de Scikit-Learn.")

# ------------------------------------------------------------------------------
# 1. EJECUCIÓN DEL MCA
# ------------------------------------------------------------------------------
# Preparamos las columnas categóricas originales (rellenando nulos para la tabla disyuntiva)
X_cat_clean = X_cat.fillna('VALOR_FALTANTE_NA').astype(str)

if usar_prince:
    mca = prince.MCA(n_components=10, random_state=42)
    mca.fit(X_cat_clean)
    
    inercia_dim = mca.percentage_of_variance_
    inercia_1 = inercia_dim[0]
    inercia_2 = inercia_dim[1]
    
    # Coordenadas de las categorías
    coords_cats = mca.column_coordinates(X_cat_clean)
    coords_cats.columns = [f'Dim{i+1}' for i in range(coords_cats.shape[1])]
    
    # Calidad de representación (cos2 aprox)
    # ❌ LÍNEA ELIMINADA: contrib = mca.column_contributions(X_cat_clean)
    # La eliminamos por incompatibilidad de versión, ya que calculamos cos2 directamente abajo:
    
    cos2_df = coords_cats**2
    cos2_df['calidad_plano_1_2'] = cos2_df['Dim1'] + cos2_df['Dim2']
    
else:
    # Motor de cálculo alternativo por SVD sobre tabla de contingencia disyuntiva
    from sklearn.decomposition import TruncatedSVD
    
    X_dummies = pd.get_dummies(X_cat_clean, dtype=float)
    svd_mca = TruncatedSVD(n_components=10, random_state=42)
    svd_mca.fit(X_dummies)
    
    inercia_1 = svd_mca.explained_variance_ratio_[0] * 100
    inercia_2 = svd_mca.explained_variance_ratio_[1] * 100
    
    coords_cats = pd.DataFrame(
        svd_mca.components_.T[:, :2], 
        index=X_dummies.columns, 
        columns=['Dim1', 'Dim2']
    )
    cos2_df = coords_cats**2
    cos2_df['calidad_plano_1_2'] = cos2_df['Dim1'] + cos2_df['Dim2']

print("\n" + "="*75)
print("📊 RESULTADOS DEL DIAGNÓSTICO MCA (INERCIA CUALITATIVA):")
print(f"· Inercia Explicada Dimensión 1: {inercia_1:.2f}%")
print(f"· Inercia Explicada Dimensión 2: {inercia_2:.2f}%")
print(f"· Inercia Acumulada Plano 2D:  {inercia_1 + inercia_2:.2f}%")
print("-" * 75)

# Top categorías mejor y peor representadas en el mapa 2D (cos2)
top_bien_rep = cos2_df.sort_values(by='calidad_plano_1_2', ascending=False).head(5)
top_mal_rep = cos2_df.sort_values(by='calidad_plano_1_2', ascending=True).head(5)

print("\n🌟 Categorías MEJOR representadas en el mapa 2D (cos² alto):")
display(top_bien_rep[['Dim1', 'Dim2', 'calidad_plano_1_2']])

print("\n⚠️ Categorías CON BAJA representación en 2D (no interpretar posición aislada):")
display(top_mal_rep[['Dim1', 'Dim2', 'calidad_plano_1_2']])

# ------------------------------------------------------------------------------
# 2. GUARDADO DE ARTEFACTOS Y REPORTES JSON
# ------------------------------------------------------------------------------
os.makedirs('artifacts', exist_ok=True)
os.makedirs('reports', exist_ok=True)

# Guardamos el objeto MCA en artifacts
if usar_prince:
    joblib.dump(mca, 'artifacts/mca.joblib')

reporte_pca_mca = {
    'pca_varianza_plano_2d_pct': round(float(var_explicada[0]+var_explicada[1])*100, 2),
    'pca_componentes_80_pct': int(comp_80),
    'pca_componentes_90_pct': int(comp_90),
    'mca_inercia_plano_2d_pct': round(float(inercia_1 + inercia_2), 2),
    'mca_top_categorias_mejor_representadas': list(top_bien_rep.index)
}

with open('reports/pca_mca.json', 'w') as f:
    json.dump(reporte_pca_mca, f, indent=4)

print("\n" + "="*75)
print("✅ ARTEFACTOS DE FACTORES FACTORIALES EXPORTADOS:")
print("· Modelo MCA guardado en: 'artifacts/mca.joblib'")
print("· Reporte de Diagnóstico guardado en: 'reports/pca_mca.json'")
print("="*75)


📊 RESULTADOS DEL DIAGNÓSTICO MCA (INERCIA CUALITATIVA):
· Inercia Explicada Dimensión 1: 3.15%
· Inercia Explicada Dimensión 2: 2.64%
· Inercia Acumulada Plano 2D:  5.79%
---------------------------------------------------------------------------

🌟 Categorías MEJOR representadas en el mapa 2D (cos² alto):


,Dim1,Dim2,calidad_plano_1_2
actual_time__K-12_Student,0.0136,5.8319,5.8455
primary_place_of_www_access__School,0.1220,4.5491,4.6711
who_pays_for_access_parents__Si,0.0040,4.1529,4.1570
education_attainment__Grammar,0.0439,4.0498,4.0937
age__< 18 años,0.1080,3.6559,3.7640



⚠️ Categorías CON BAJA representación en 2D (no interpretar posición aislada):


,Dim1,Dim2,calidad_plano_1_2
community_membership_religious__No,0.0000,0.0000,0.0001
not_purchasing_other__No,0.0000,0.0000,0.0001
sexual_preference__Heterosexual,0.0002,0.0000,0.0002
how_you_heard_about_survey_others__No,0.0000,0.0003,0.0003
not_purchasing_too_complicated__No,0.0001,0.0003,0.0004



✅ ARTEFACTOS DE FACTORES FACTORIALES EXPORTADOS:
· Modelo MCA guardado en: 'artifacts/mca.joblib'
· Reporte de Diagnóstico guardado en: 'reports/pca_mca.json'


## 4. Conclusiones del Diagnóstico Factorial (PCA y MCA)

El análisis factorial no supervisado sobre la estructura espacial de los datos aporta los siguientes hallazgos estratégicos:

---

### A. Diagnóstico de Dimensionalidad (PCA sobre Matriz Procesada)
* **Alta Dispersión de Varianza:** El plano 2D de PCA únicamente captura un **$12.63\%$ de la varianza total** ($7.43\%$ Dim 1 + $5.20\%$ Dim 2). Se requieren **$56$ componentes principales** para acumular el $80\%$ de varianza.
* **Justificación de No Inclusión en Modelado Supervisado:** En modelos de negocio, regulación o riesgo, no se recomienda entrenar con componentes principales de PCA, ya que destruyen la interpretabilidad de cada predictor frente al cliente. Su función aquí es estrictamente diagnóstica.

---

### B. Mapeo Perceptual y Representación ($cos^2$ en MCA)
* **Inercia Acumulada:** El plano 2D del MCA captura un **$5.79\%$ de la inercia cualitativa total** ($3.15\%$ Dim 1 + $2.64\%$ Dim 2).
* **Perfiles Extremos Identificados (cos² alto):** Las categorías con mayor fidelidad de representación ($cos^2$) en los ejes 2D corresponden al grupo demográfico escolar y dependiente:
  * `actual_time__K-12_Student`
  * `primary_place_of_www_access__School`
  * `who_pays_for_access_parents__Si`
  * `education_attainment__Grammar`
  * `age__< 18 años`
  * **Insight de Negocio:** El segmento escolar/menor de edad es geométricamente el más polarizado en el espacio, separándose marcadamente de la masa de usuarios adultos/profesionales.

---

### 🏁 Transición al Cuaderno 09 (Balanceo de Datos):
Con la estructura espacial diagnosticada, procedemos al cuaderno **`09_balanceo.ipynb`**, donde evaluaremos la distribución del Target (`Cluster`), probaremos técnicas de muestreo (**SMOTE, RandomOverSampler, RandomUnderSampler, Class Weight**) y determinaremos si el balanceo sintético mejora la capacidad discriminante del modelo.